In [50]:
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
import rasterio

IN_FILE  = Path("datasets/resultset.xlsx")
OUT_FILE = Path("datasets/fineli_head_medians.xlsx")
HEAD_FINELLI_NO_LEMMATIZATION = Path("datasets/fineli_head_medians_no_lemmatization.xlsx")
NUMBERS_FILE = Path("datasets/fineli_pre_processed.xlsx")

LESS_THAN_REPLACEMENT_VALUE = 0.05  # "<0.1" is treated as 0.05

In [51]:
def to_number(value):
    """'12.5' -> 12.5, '<0.1' -> 0.05, 'N/A' or empty -> NaN"""
    if pd.isna(value):
        return np.nan
    value = str(value).strip().replace(',', '.')  # In case '12,5' -> '12.5'
    if value.upper() in {'N/A', 'NA', '-', ''}:
        return np.nan
    if value.startswith('<'):  # '<0.1' -> '0.05'
        return LESS_THAN_REPLACEMENT_VALUE
    try:
        return float(value)
    except ValueError:
        return np.nan


def head_term(name):
    """'Ahven, paistettu voissa' -> 'ahven'"""
    head = str(name).split(',')[0]  # text before the first comma
    head = head.split('/')[0]  # first alternative only, 'Aromisuola/yrttisuola 71 %' -> 'aromisuola'
    head = re.split(r'\d', head)[0]  # cut at the first number, 'kasvirasvalevite 70 %' -> 'kasvirasvalevite'
    head = unicodedata.normalize('NFC', head)  # consistent å/ä/ö for later matching
    head = re.sub(r'\s+', ' ', head).strip().lower()  # replace multiple whitespaces with single whitespaceS
    return head

In [52]:
import stanza  # TODO: use better lemmatizer????

stanza.download('fi')
nlp = stanza.Pipeline('fi', processors='tokenize,pos,lemma')

2026-10-13 16:29:34 INFO: Downloaded file to C:\Users\kaspe\AppData\Local\StanfordNLP\stanza\Cache\1.15.0\resources\resources.json
2026-10-13 16:29:34 INFO: Downloading default packages for language: fi (Finnish) ...
2026-10-13 16:29:35 INFO: File exists: C:\Users\kaspe\AppData\Local\StanfordNLP\stanza\Cache\1.15.0\resources\fi\default.zip
2026-10-13 16:29:38 INFO: Finished downloading models and saved to C:\Users\kaspe\AppData\Local\StanfordNLP\stanza\Cache\1.15.0\resources
2026-10-13 16:29:38 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-10-13 16:29:38 INFO: Downloaded file to C:\Users\kaspe\AppData\Local\StanfordNLP\stanza\Cache\1.15.0\resources\resources.json
2026-10-13 16:29:38 WARNING: Language fi package default expects mwt, which has been added
2026-10-13 16:29:39 INFO: Loading these models for language: fi (Finnish):
| Process

In [53]:
def clean_lemma(word):
    """Remove the compound marker: 'kaura#puuro' -> 'kaurapuuro',
    'ateria#aines' (from 'ateria-ainekset') -> 'ateria-aines'
    """
    lemma = word.lemma or word.text
    return lemma.replace('#', '-' if '-' in word.text else '')

def lemmatize_term(term):
    """'ahvenet foliossa' -> 'ahven folio'"""
    if not term:
        return term
    doc = nlp(term)
    return ' '.join(clean_lemma(w) for s in doc.sentences for w in s.words).lower()

print(lemmatize_term('ahvenet foliossa'))  # expect: ahven folio
print(lemmatize_term('herneet'))  # expect: herne
print(lemmatize_term('ateria-ainekset'))  # check how the hyphen is handled (expect: ateria-aines)

ahven folio
herne
ateria-aines


In [54]:
def read_fineli_raw(in_file):
    return pd.read_excel(in_file, dtype=str)

def convert_to_numbers(raw):
    nutrient_cols = [c for c in raw.columns if c not in ('id', 'name')]
    numbers = raw.copy()
    numbers[nutrient_cols] = raw[nutrient_cols].apply(lambda col: col.map(to_number))
    return numbers

def calculate_medians(numbers):
    df = numbers.drop(columns='id')
    df['name'] = df['name'].map(head_term)

    lemma_map = {head: lemmatize_term(head) for head in df['name'].unique()}  # unique only = fast
    df['name'] = df['name'].map(lemma_map)

    return df.groupby('name').median()   # 'ahvenet' and 'ahven' now share one row

def calculate_medians_no_lemmatization(numbers):
    df = numbers.drop(columns='id')
    df['name'] = df['name'].map(head_term)
    return df.groupby('name').median()

def load_fineli_dataset(in_file=IN_FILE, numbers_file=NUMBERS_FILE, out_file=OUT_FILE):
    raw = read_fineli_raw(in_file)

    numbers = convert_to_numbers(raw)
    # numbers.to_excel(numbers_file, index=False)

    medians = calculate_medians(numbers)
    medians.to_excel(out_file)
    return medians

In [55]:
fineli = load_fineli_dataset()
print("\n", fineli)


                     energia, laskennallinen (kJ)  rasva (g)  \
name                                                          
ahven folio                           495.981167   4.080596   
ahvi                                  597.447244   6.642549   
amarantti                            1545.439941   7.000000   
ananas                                270.299995   0.306000   
ananassa                              208.769466   0.358155   
...                                          ...        ...   
wokattu kasvis                        362.226265   6.849243   
yrtti                                 108.538630   0.513268   
yrttilevite                          1589.106688  40.113636   
äidinmaidonkorvike                   1180.312120  14.262000   
äidinmaito                            273.049988   3.495000   

                    hiilihydraatti imeytyvä (g)  proteiini (g)  alkoholi (g)  \
name                                                                           
ahven folio       

In [56]:

def loadSuomi24Corpus(path):
    """More like instantiate the shit"""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        n=24
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line)  
        
def retrieve_threads_and_messages():
    """Retrieve threads and messages from suomi24 dataset by using fineli dataset. Return messages and threads where match was found. Return also date"""
    pass


def draw_monthly_nutrition_intake():
    """Draw montly intake evolution of calories on foods"""
    pass

loadSuomi24Corpus('datasets/s24_2001.vrt')



<!-- #vrt positional-attributes: word ref lemma lemmacomp pos msd dephead deprel spaces initid lex/ -->

<!-- #vrt info: VRT generated from CWB data for corpus "s24_2001" (2020-03-05 23:45:53 +0200) -->

<!-- #vrt info: A processing log at the end of file -->

<text comment_id="0" date="2001-01-01" datetime="2001-01-01 01:30:00" author="Honda" parent_comment_id="0" quoted_comment_id="0" author_logged_in="n" nick_type="anonymous" thread_id="19455" time="01:30:00" title="Hyvää uutta vuotta kaikille Hondailijoille" topic_nums="3258,1109,6254,2" msg_type="thread_start" topic_name_leaf="Honda" topic_name_top="Ajoneuvot ja liikenne" topic_names="Ajoneuvot ja liikenne &gt; Autot &gt; Automerkit &gt; Honda" topic_names_set="|Ajoneuvot ja liikenne|Automerkit|Autot|Honda|" topic_nums_set="|1109|2|3258|6254|" topic_adultonly="n" datefrom="20010101" dateto="20010101" timefrom="013000" timeto="013000" id="19455:0" author_v1="Honda" author_name_type="user_nickname" author_nick_registered="n" author_